In [ ]:
import anndata as ad
# adata = ad.read_h5ad("./data/larry/postprocessed.h5ad")
adata = ad.read_h5ad("./data/larry/larry_processed.h5ad")
adata

In [ ]:
import joblib
from scripts.VectorFieldEmbedder import VectorFieldEmbedder
from scripts.plotting import *

# Load the object back
emb = joblib.load("./data/larry/larry_embedder.pkl")
emb.gene_names = adata.var_names
emb.cell_state = list(adata.obs["state_info"])

print("Loaded object:", type(emb))

In [ ]:
def embedding_dist(Z, i_idx, j_idx):
    return np.linalg.norm(Z[i_idx] - Z[j_idx], axis=1)


def binned_corr(d_ref, d_test, n_bins=10, n_per_bin=4000, rng=np.random.default_rng(0)):
    qs = np.linspace(0, 1, n_bins + 1)
    edges = np.quantile(d_ref, qs)

    corrs = []
    centers = []

    for b in range(n_bins):
        lo, hi = edges[b], edges[b + 1]
        idx = np.where((d_ref >= lo) & (d_ref <= hi))[0]

        if len(idx) == 0:
            corrs.append(np.nan)
            centers.append((lo + hi) / 2)
            continue

        take = min(n_per_bin, len(idx))
        sub = rng.choice(idx, size=take, replace=False)

        corrs.append(pearsonr(d_ref[sub], d_test[sub])[0])
        centers.append(np.median(d_ref[sub]))

    return np.array(centers), np.array(corrs)

In [ ]:
from sklearn.decomposition import PCA

X = emb.X
rng = np.random.default_rng(0)
n, d = X.shape

k_per_point = 400

# --------------------------------------------------
# Sample random pairs
# --------------------------------------------------
i_idx = np.repeat(np.arange(n), k_per_point)
j_idx = rng.integers(0, n, size=n * k_per_point)

mask = i_idx != j_idx
i_idx = i_idx[mask]
j_idx = j_idx[mask]

# --------------------------------------------------
# Compute raw distances ONCE
# --------------------------------------------------
diff = X[i_idx] - X[j_idx]        # (N_pairs, d)
d_raw = np.linalg.norm(diff, axis=1)

In [ ]:
from scipy.stats import pearsonr

curves = {}

# ---- PCA baselines (generate PCA explicitly) ----
pca_dims = [2, 4, 8, 16]

for d in pca_dims:
    print(f"Computing PCA-{d}")
    
    Z_pca = PCA(n_components=d, random_state=0).fit_transform(X)
    d_pca = embedding_dist(Z_pca, i_idx, j_idx)
    
    centers, c = binned_corr(d_raw, d_pca, rng=rng)
    curves[f"PCA-{d}"] = (centers, c)

# ---- UMAP 2D (already computed) ----
print("Computing UMAP-2")

Z_umap = emb.X_emb              # shape (n_cells, 2)
d_umap = embedding_dist(Z_umap, i_idx, j_idx)

centers, c_umap = binned_corr(d_raw, d_umap, rng=rng)
curves["UMAP-2"] = (centers, c_umap)

# ---- UMAP 2D + TPS ----
print("Computing UMAP-2 + TPS")

Z_umap_tps = emb.tps.predict(Z_umap)
d_umap_tps = embedding_dist(Z_umap_tps, i_idx, j_idx)

centers, c_umap_tps = binned_corr(d_raw, d_umap_tps, rng=rng)
curves["UMAP-2+TPS"] = (centers, c_umap_tps)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(5.5, 4.5))

# x-axis = distance quantile
n_bins = len(curves["UMAP-2"][0])
xq = (np.arange(n_bins) + 0.5) / n_bins

# PCA (dashed)
for d in pca_dims:
    _, c = curves[f"PCA-{d}"]
    plt.plot(
        xq, c,
        linestyle="--",
        linewidth=1.8,
        alpha=0.8,
        label=f"PCA-{d}",
    )

# UMAP
_, c = curves["UMAP-2"]
plt.plot(
    xq, c,
    linewidth=2.2,
    label="UMAP-2",
)

# UMAP + TPS
_, c = curves["UMAP-2+TPS"]
plt.plot(
    xq, c,
    linewidth=2.8,
    label="UMAP-2 + TPS",
)

plt.xlabel("Distance quantile")
plt.ylabel("Pearson correlation\nwithin distance bin")
plt.ylim(-0.2, 1.0)

plt.xticks(
    [0.1, 0.3, 0.5, 0.7, 0.9],
    ["10%", "30%", "50%", "70%", "90%"],
)

plt.legend(frameon=False, fontsize=9)
plt.title("Metric fidelity across distance scales", fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
from scipy.spatial.distance import cdist
from scipy.stats import spearmanr, rankdata

cell_state = np.array(emb.cell_state)

X_ambient = emb.X
X_umap = emb.X_emb
X_tps = emb.tps.predict(X_umap)
labels = cell_state

In [ ]:
from scipy.spatial.distance import cdist
from scipy.stats import kendalltau
import numpy as np

def celltype_distance_matrix(X, labels, metric="euclidean"):
    types = np.unique(labels)
    D = np.zeros((len(types), len(types)))

    for i, ti in enumerate(types):
        Xi = X[labels == ti]
        for j, tj in enumerate(types):
            if i == j:
                continue
            Xj = X[labels == tj]
            D[i, j] = cdist(Xi, Xj, metric=metric).mean()
    return D, types


def kendall_per_type(D_ref, D_emb):
    taus = []
    for i in range(D_ref.shape[0]):
        # exclude self-distance
        ref = np.delete(D_ref[i], i)
        emb = np.delete(D_emb[i], i)

        tau, _ = kendalltau(ref, emb)
        taus.append(tau)
    return np.array(taus)

In [ ]:
D_ambient, types = celltype_distance_matrix(emb.X, labels, metric="euclidean")
D_umap, _       = celltype_distance_matrix(X_umap, labels, metric="euclidean")
D_tps, _        = celltype_distance_matrix(X_tps, labels, metric="euclidean")

taus_umap = kendall_per_type(D_ambient, D_umap)
taus_tps  = kendall_per_type(D_ambient, D_tps)

print(taus)
print(taus_tps)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = [taus, taus_tps]
labels = ["UMAP", "UMAP + TPS"]

plt.figure(figsize=(5, 4))

# boxplot
plt.boxplot(
    data,
    labels=labels,
    showfliers=False,
    widths=0.6
)

# jittered scatter
for i, vals in enumerate(data, start=1):
    x = np.random.normal(i, 0.04, size=len(vals))  # jitter
    plt.scatter(
        x,
        vals,
        alpha=0.7,
        s=25,
        color="black",
        zorder=3
    )

plt.ylabel("Kendall's Tau (cell-type neighbor ranking)")
plt.title("Global cell-type geometry preservation")
plt.axhline(0, color="gray", lw=1, linestyle="--")
plt.tight_layout()
plt.show()